# Running the CESM2 diffusion emulator

A step-by-step guide, from "I have a checkpoint" to "I have a figure".

**Two ways to run the emulator, and this notebook covers both:**

| | what it is | where it runs | when to use it |
|---|---|---|---|
| **Path A** | `eval_aero.py` via SLURM | LUMI, 4–7 GPUs | the normal way: every scenario, many members, writes NetCDF + plots |
| **Path B** | call the model directly in Python | any single GPU | understanding the pipeline, one-off experiments, debugging |

**Path C**, at the end, is training.

### Cell colour code

Cells are tagged in their first comment line:

- `# LOCAL` — runs on your workstation, no GPU, reads over the sshfs mounts
- `# LUMI` — needs a GPU and the LUMI filesystem; run it in a batch job
- `# SHELL` — a command to run in a terminal, shown for copying

Nothing here runs a job by itself unless you execute the cell.

---

## What the emulator actually is

A **conditional video diffusion model**. It denoises a `(batch, channels, 1, 192, 288)`
field, conditioned on gridded forcing maps. For the `run_mseyb_BCprect` run:

- **Input (conditioning)**: 3 channels — cumulative CO₂, sulphate (SUL), black carbon (BC),
  each a `192×288` map per year
- **Output (target)**: 2 channels — `TREFHT` (surface air temperature) and `PRECT`
  (total precipitation)
- **One sample = one year.** A scenario is generated year by year; an ensemble
  member is one seed.

So "running the emulator" means: load a checkpoint, prepare conditioning maps
for the years you want, sample, and denormalise.

---
# Step 0 — What you need, and where it lives

Three artefacts. If you have all three you can run.

| artefact | example | note |
|---|---|---|
| **checkpoint** | `runs/run_mseyb_BCprect_860.pt` | contains `EMA` weights, `PCA` basis, `COND_NORM` ranges |
| **model config** | `configs/config_aero.yaml` | defines the UNet and the noise scheduler |
| **conditioning file** | `emissions_ssp370_only_timefixed_bc_co2fix.nc` | CO2/SUL/BC maps per year |

Plus, only if you want to *score* the output: a CESM2 reference (the training
tree or `cmip6/*.nc`).

### Projects and paths — read this before anything else

Two LUMI projects are in play and mixing them up is the most common failure:

| | project | scratch mount (local) |
|---|---|---|
| data, cond files, venv, most training | **462001328** | `~/mnt/lumi_sc2` |
| the ep0860 training run + all eval output | **462001112** | `~/mnt/lumi_sc` |

`lumi_env.sh` is the single source of truth: `LUMI_PROJECT` picks the data/account,
`LUMI_EVAL_PROJECT` picks where eval output lands. **The `ep0860` run lives on
462001112**, so anything touching it needs `LUMI_PROJECT=462001112`.

> ⚠️ **The sshfs mounts serve stale file *contents*.** A directory listing is
> usually fine, but never judge what code is deployed on LUMI from a mount read —
> use `ssh` and `git log`.

In [ ]:
# LOCAL — check the mounts are alive and find the newest checkpoints.
# An EMPTY listing means the mount is dead, not that the directory is empty.
import subprocess, textwrap

print(subprocess.run(
    "ls /home/nordling/mnt/lumi_sc/eval_output/run_mseyb_BCprect/ | tail -5",
    shell=True, capture_output=True, text=True).stdout)

# Checkpoints for the ep0860 run live on project 462001112's projappl
print(subprocess.run(
    "ls -lat /home/nordling/mnt/lumi/CESM2_emulator_from_lumi/runs/run_mseyb_BCprect_*.pt "
    "| head -5", shell=True, capture_output=True, text=True).stdout)

---
# Path A — the normal way: `eval_aero.py` on SLURM

This is what produced every figure in the paper. It generates each scenario,
compares against CESM2, and writes one NetCDF per (variable, scenario) plus
summary plots.

### A.1 Submit

`submit_eval_ens25.sh` wraps `run_eval_aero.sh` with sensible resources. It
takes `<checkpoint> [experiments] [output_subdir]`:

In [ ]:
# SHELL — submit a 25-member evaluation of ep0860 (this is job 21538699 verbatim)
#
# NOTE the two things that are easy to get wrong:
#   LUMI_PROJECT=462001112  -> this run's data/account, NOT the 328 default
#   NTASKS=7                -> 8 tasks x 8 cpus = 64 cores, but LUMI-G reserves 8,
#                              so 8 ranks is rejected with
#                              "Requested node configuration is not available"
#
# ssh nordlin1@lumi.csc.fi
# cd /projappl/project_462001112/CESM2_emulator_from_lumi
# LUMI_PROJECT=462001112 MEMBERS=25 NTASKS=7 WALLTIME=10:00:00 \
#     bash submit_eval_ens25.sh runs/run_mseyb_BCprect_860.pt "" ep0860_ens25
#
# Sharding is per EXPERIMENT, so walltime is set by the single heaviest one
# (aaer, 201 years) on one GPU: ~6.7 h at 25 members. More ranks than
# experiments just idles GPUs.
#
# For a quick 5-member run of one scenario instead:
# EXPERIMENTS=ssp370 MEMBERS=5 bash submit_eval_ens25.sh runs/run_mseyb_BCprect_860.pt

### A.2 Watch it

```bash
squeue -u nordlin1
tail -f logs/eval_ens25_<jobid>.out
```

Two log lines tell you it is healthy:

- `[EVAL-CFG] CHECKPOINT=... MEMBERS=25` — confirms it loaded what you meant
- `[SHARD] rank=i/N running=[...]` — the experiment split across GPUs

> ⚠️ **A "COMPLETED" job can still have failed at plotting.** The 462001112 venv
> has a broken cartopy/shapely: map plotting dies with
> `GEOSException: Points of LinearRing do not form a closed linestring` *after*
> the NetCDFs are written. The data is fine; re-render figures locally.

### A.3 What you get

```
eval_output/manual/ep0860_ens25/
├── TREFHT_hist.nc        TREFHT_ssp370.nc     TREFHT_aaer.nc  ...
├── PRECT_hist.nc         PRECT_ssp370.nc      ...
├── global_mean_anomaly.csv / .png
└── _shards/rank*.pkl     (per-rank intermediates, rank 0 merges them)
```

Each NetCDF holds, per member `m1…mN`:
`<VAR>_model_m<N>` (absolute), `<VAR>_model_m<N>_anom` (vs 1850–1900),
`<VAR>_model_gmean_m<N>`, and the same four for `<VAR>_cesm_*`.

In [ ]:
# LOCAL — inspect an eval NetCDF without a GPU. This is how every figure script starts.
import xarray as xr, numpy as np, re

EVAL = "/home/nordling/mnt/lumi_sc/eval_output/manual/ep0860_ens25"
ds = xr.open_dataset(f"{EVAL}/TREFHT_ssp370.nc")

emu  = sorted([v for v in ds.data_vars if re.fullmatch(r"TREFHT_model_gmean_m\d+", v)],
              key=lambda x: int(x.rsplit("_m", 1)[1]))
cesm = sorted([v for v in ds.data_vars if re.fullmatch(r"TREFHT_cesm_gmean_m\d+", v)],
              key=lambda x: int(x.rsplit("_m", 1)[1]))
print(f"{len(emu)} emulator members, {len(cesm)} CESM2 members, "
      f"years {ds.year.values.min()}-{ds.year.values.max()}")

E = np.stack([ds[v].values for v in emu])      # (member, year), absolute degC
C = np.stack([ds[v].values for v in cesm])
print(f"2091-2100 global mean:  emulator {E[:, -10:].mean():.3f} degC   "
      f"CESM2 {C[:, -10:].mean():.3f} degC")

---
# Path B — running the emulator by hand

Everything `eval_aero.py` does, in six cells. **This needs a GPU.** Run it inside
a batch job or an interactive allocation on LUMI:

```bash
srun --account=project_462001112 --partition=small-g --gpus-per-node=1 \
     --cpus-per-task=8 --mem=64G --time=1:00:00 --pty bash
source lumi_env.sh && source $LUMI_VENV/bin/activate
```

The steps are: **load model → build conditioning → sample → denormalise**.

In [ ]:
# LUMI — B.1  Load the checkpoint.
#
# load_model() returns the EMA weights (not the raw ones) and the PCA state, and
# it injects the checkpoint's COND_NORM clip ranges into the dataset module so
# that inference normalises conditioning EXACTLY as training did.
import torch, lumi_paths as L
from omegaconf import OmegaConf
from hydra.utils import instantiate
from eval_aero import load_model, build_cond_tensor, generate_timeseries

CKPT   = "/projappl/project_462001112/CESM2_emulator_from_lumi/runs/run_mseyb_BCprect_860.pt"
MCFG   = "configs/config_aero.yaml"
device = torch.device("cuda")

model, pca_state = load_model(CKPT, MCFG, device)

# Watch for this in the output:
#   [COND-NORM] using checkpoint-persisted clip ranges   <- good
#   [COND-NORM] WARNING: checkpoint has no COND_NORM      <- MISCALIBRATED, stop
print("PCA state:", None if pca_state is None else list(pca_state.keys()))

In [ ]:
# LUMI — B.2  The noise scheduler comes from the same config as the model.
cfg       = L.resolve_cfg(OmegaConf.load(MCFG))
scheduler = instantiate(cfg.scheduler)          # ContinuousDDPM
OUT_CH    = int(cfg.model.get("out_channels", 1))
print(f"out_channels = {OUT_CH}   (1 = TREFHT only, 2 = TREFHT + PRECT)")

In [ ]:
# LUMI — B.3  Build the conditioning tensor.
#
# THE CRITICAL STEP. Smoothing and PCA must mirror training exactly; feed the
# model raw inventory fields it never saw and it imprints grid-scale texture
# (shipping lanes, flight paths) onto the output.
#
# pca_objects:
#   pca_state["cond"] -> APPLY the persisted basis   (trained scenarios)
#   "fit"             -> FIT a fresh basis on this cond (unseen, e.g. ssp126)
#   None              -> SKIP PCA  (almost never what you want)
DCFG = L.resolve_cfg(OmegaConf.load("configs/config_data_ybias_BCprect.yaml"))
COND = ("/scratch/project_462001112/emulator_data/"
        "emissions_ssp370_only_timefixed_bc_co2fix.nc")

cond_tensor, years, lat, lon = build_cond_tensor(
    cond_file           = COND,
    cond_vars           = list(DCFG.cond_vars),            # ["CO2", "SUL", "BC"]
    time_dim            = "time",
    pca_objects         = pca_state.get("cond") if pca_state else None,
    n_components_cond   = list(DCFG.n_components_cond),    # [30, 5, 5]
    cond_smooth_sigma   = list(DCFG.cond_smooth_sigma),    # [0, 2, 2]
    cond_smooth_method  = DCFG.get("cond_smooth_method", "gaussian"),
)
print(f"cond {tuple(cond_tensor.shape)} = (n_vars, T, H, W), "
      f"years {years.min()}-{years.max()}")

In [ ]:
# LUMI — B.4  Sample. One diffusion run over every year in cond_tensor.
#
# seed  -> the ensemble member. Same cond + different seed = different member.
# target_channel=None returns ALL channels, so one pass feeds both TREFHT and
# PRECT rather than sampling twice.
gen_norm = generate_timeseries(
    model         = model,
    scheduler     = scheduler,
    cond_tensor   = cond_tensor,
    device        = device,
    dtype         = torch.bfloat16,      # bf16 is ~2x faster and matches the eval
    sample_steps  = 50,                  # production default; 200 for a sampler test
    batch_size    = 16,
    seed          = 1234,
    guidance_co2  = 1.0,                 # 1.0 everywhere = direct conditioning,
    guidance_sul  = 1.0,                 # ONE forward pass. Anything else turns on
    guidance_bc   = 1.0,                 # per-channel CFG and costs 4 passes.
    out_channels  = OUT_CH,
    target_channel= None,
)
print(gen_norm.shape, "= (T, C, H, W), still in NORMALISED model space")

In [ ]:
# LUMI — B.5  Denormalise. The sampler returns normalised space; each target
# channel has its own inverse transform.
from data.climate_dataset import DENORM_FN
import numpy as np

target_vars = list(DCFG.target_vars)             # ["TREFHT", "PRECT"]
fields = {v: DENORM_FN[v](gen_norm[:, i]) for i, v in enumerate(target_vars)}

w = np.cos(np.deg2rad(lat))[None, :, None]
for v, arr in fields.items():
    g = (arr * w).sum(axis=(1, 2)) / (w.sum() * arr.shape[2])
    print(f"{v:7s} {arr.shape}  global mean {g[0]:.3f} (first yr) -> "
          f"{g[-1]:.3f} (last yr)")

In [ ]:
# LUMI — B.6  An ensemble is just a loop over seeds.
members = {}
for seed in (1234, 1235, 1236):
    g = generate_timeseries(model, scheduler, cond_tensor, device, torch.bfloat16,
                            sample_steps=50, batch_size=16, seed=seed,
                            out_channels=OUT_CH, target_channel=0)   # 0 = TREFHT
    members[seed] = DENORM_FN["TREFHT"](g)
print(f"{len(members)} members of shape {next(iter(members.values())).shape}")

# Cost: ~1 GPU-hour per 200 scenario-years per member at 50 steps. This is why
# 25 members x 9 experiments is a ~7 h job on 7 GPUs.

---
# Step 4 — Making the paper figures

All figure scripts read the eval NetCDFs, so they run **locally** — but they need
cartopy, which is only in the `plotting` conda env:

```bash
PY=/home/nordling/miniconda3/envs/plotting/bin/python
```

> ⚠️ `paper_fig_maps.py` **refuses to run** (exit 2) without cartopy unless you
> pass `--allow-no-cartopy`. That guard exists because running it in the base env
> once overwrote a good figure with flat lat/lon panels.

Two flags matter for every script:

| flag | why |
|---|---|
| `--n-ref-members 0` | use **all** held-out CESM2 members (10/10/11/6), not the default 5 |
| `--match-members` | cap the emulator at the reference count, so neither mean is better converged |

In [ ]:
# SHELL — regenerate the figure set at equal sampling
# PY=/home/nordling/miniconda3/envs/plotting/bin/python
# EVAL=/home/nordling/mnt/lumi_sc/eval_output/manual/ep0860_ens25
# DATA=/home/nordling/mnt/lumi_sc/emulator_data
#
# $PY scripts/paper_fig_timeseries.py  --var TREFHT --eval-dir $EVAL --data-root $DATA \
#     --n-ref-members 0 --match-members --no-emu-spread --out plots/fig01.png
# $PY scripts/paper_fig_maps.py        --eval-dir $EVAL --data-root $DATA \
#     --n-ref-members 0 --match-members --out plots/fig03.png --csv plots/map_stats.csv
# $PY scripts/paper_fig_attribution.py --eval-dir $EVAL --maps --decompose \
#     --match-members --out plots/fig12.png --csv plots/attribution.csv
#
# Passing a .png OUT makes every script write the .pdf sibling too.
# The maps script re-reads all CESM2 members over the mount: ~13 min per run.

---
# Path C — training

`main_aero.py` is a Hydra app: the config *is* the interface, and anything in it
can be overridden on the command line.

```bash
# on LUMI, from the repo
sbatch run_mseyb_BCprect.sh
```

Under the hood that runs, via accelerate:

```
main_aero.py --config-name=config_aero.yaml \
    trainer.hyperparameters.save_dir=... \
    trainer.hyperparameters.save_every=140
```

### The three settings that decide whether a run is usable

| key | meaning | trap |
|---|---|---|
| `save_every` | checkpoint interval in **optimizer steps**, not epochs | too large and a 36 h job saves nothing |
| `load_path` | `0` = fresh, `"newest"` = resume, or an explicit `.pt` | see below |
| `reset_optimizer` | keep Adam momentum across chained jobs | must stay `false` for self-chaining |

> ⚠️ **Resuming is deliberately off by default** in the monthly config
> (`load_path: 0`), because a resume from a mid-epoch checkpoint once skipped
> 2249 of an epoch's 2250 steps and then deadlocked. For an intentional
> continuation, pass the checkpoint explicitly and **write to a fresh
> `save_dir`** so rotation cannot delete the checkpoints you are resuming from
> (only the 5 newest are kept).

```bash
sbatch --account=project_2019839 --partition=gpularge --nodes=1 \
       --ntasks-per-node=4 --gres=gpu:gh200:4 --time=1-12:00:00 \
       run_roihu.sh main_aero.py --config-name=config_aero_monthly.yaml \
       trainer.hyperparameters.save_every=140 \
       trainer.hyperparameters.save_dir=/scratch/project_2019839/runs/0826_cont \
       trainer.hyperparameters.load_path=/scratch/.../run_monthly_bcprect_163.pt
```

**Verify a resume is healthy** by watching the first two epochs: the first is a
fast-forward through the dataloader (~2 min instead of ~13), the second must do
real work at a normal step time, and a checkpoint must appear within
`save_every` optimizer steps. Silence with GPUs pinned at 100% is the deadlock.

---
# Gotchas — the list worth re-reading

**Checkpoint / conditioning**
1. `COND_NORM` missing from a checkpoint ⇒ the eval is **miscalibrated**, not merely approximate. The loader warns; do not ignore it.
2. PCA basis: apply the persisted one for trained scenarios, `"fit"` for unseen ones. Skipping PCA feeds full-rank cond the model never saw.
3. `guidance_* = 1.0` is one forward pass; any other value costs four. CFG tuning was tried and **does not work** at high forcing — it is sub-additive.

**Paths / projects**
4. `LUMI_PROJECT` (data + account) and `LUMI_EVAL_PROJECT` (output) are separate knobs. The ep0860 run is on **462001112**.
5. Mounts serve **stale contents**. Verify deployed code over `ssh`, never from `~/mnt`.
6. On Roihu, `gpumedium` currently has `MaxSubmit = 0` — use `gpularge`. And `module` is undefined under a bare `sbatch` over ssh.

**Figures**
7. Run them in the `plotting` conda env; the base env has no cartopy.
8. `--n-ref-members` still defaults to **5** — pass `0` or you discard most of a large ensemble's statistical power.
9. Significance hatching marks where the emulator **differs from CESM2 beyond noise**: more hatching = more detectable bias, not a better score.

**Interpretation**
10. Precipitation is ~10× noisier relative to its forced signal than temperature. A blank significance map is a **detection limit**, not a pass — check the raw pre-FDR column.
11. The one-step training loss is a poor proxy for sampled-field quality; eval metrics plateau long before it does.

---
# Where to look next

| you want | file |
|---|---|
| the model | `models/video_net.py` (`UNetModel3D`) |
| the training loop | `trainer/unetTrainer.py` |
| normalisation, PCA, dataset | `data/climate_dataset.py` |
| evaluation end to end | `eval_aero.py` |
| monthly-resolution work | branch `monthly-temporal`, `eval_monthly.py` |
| the attribution maths | `figures_overleaf/attribution_equations.tex` |